# PS S6E6 - 006 LightGBM color index minimal FE

Experiment: `exp_20260603_006_lgb_color_index_min`

Purpose:
- Test minimal astronomical color-index features on top of the 002 LightGBM strict raw baseline.
- Use only competition train/test.
- Do not use original dataset.
- Do not use bias search.
- Do not tune parameters.
- Save OOF / test prediction / CV result / submission / memo / registry.

Feature Engineering:
- `u_g = u - g`
- `g_r = g - r`
- `r_i = r - i`
- `i_z = i - z`
- `u_r = u - r`
- `g_i = g - i`
- `r_z = r - z`

Date:
- 2026-06-03

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except Exception as e:
    raise ImportError("LightGBM import failed. On Kaggle, lightgbm should be available by default.") from e


COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260603_006_lgb_color_index_min"
SEED = 42
N_SPLITS = 5

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

# 002 reference
REF_002_CV = 0.9569063401002502
REF_002_PUBLIC_LB = 0.95800

print("LightGBM version:", lgb.__version__)
print("OUTDIR:", OUTDIR)

LightGBM version: 4.6.0
OUTDIR: /kaggle/working/exp_20260603_006_lgb_color_index_min


In [2]:
# ============================================================
# 1. Load data
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("sample:", sample.shape)
print("train columns:", train.columns.tolist())
print("test columns :", test.columns.tolist())
print("sample columns:", sample.columns.tolist())

display(train.head())
display(test.head())
display(sample.head())

train: (577347, 12)
test : (247435, 11)
sample: (247435, 2)
train columns: ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'class']
test columns : ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population']
sample columns: ['id', 'class']


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,GALAXY
4,577351,GALAXY


In [3]:
# ============================================================
# 2. Schema fixed from 001
# ============================================================

ID_COL = "id"
TARGET_COL = "class"

BASE_NUMERIC_COLS = [
    "alpha", "delta", "u", "g", "r", "i", "z", "redshift"
]

CATEGORICAL_COLS = [
    "spectral_type",
    "galaxy_population",
]

COLOR_FEATURES = [
    "u_g",
    "g_r",
    "r_i",
    "i_z",
    "u_r",
    "g_i",
    "r_z",
]

NUMERIC_COLS = BASE_NUMERIC_COLS + COLOR_FEATURES
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

assert ID_COL in train.columns
assert TARGET_COL in train.columns
assert ID_COL in test.columns
assert TARGET_COL not in test.columns
assert set(BASE_NUMERIC_COLS + CATEGORICAL_COLS).issubset(train.columns)
assert set(BASE_NUMERIC_COLS + CATEGORICAL_COLS).issubset(test.columns)

print("BASE_NUMERIC_COLS:", BASE_NUMERIC_COLS)
print("COLOR_FEATURES:", COLOR_FEATURES)
print("CATEGORICAL_COLS:", CATEGORICAL_COLS)

print("\\nTarget distribution:")
target_dist = train[TARGET_COL].value_counts().to_frame("count")
target_dist["rate"] = train[TARGET_COL].value_counts(normalize=True)
display(target_dist)

BASE_NUMERIC_COLS: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
COLOR_FEATURES: ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z']
CATEGORICAL_COLS: ['spectral_type', 'galaxy_population']
\nTarget distribution:


,count,rate
class,,
GALAXY,377480,0.653818
QSO,117143,0.202899
STAR,82724,0.143283


In [4]:
# ============================================================
# 3. Minimal color-index FE
# ============================================================

def add_color_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Adjacent colors
    out["u_g"] = out["u"] - out["g"]
    out["g_r"] = out["g"] - out["r"]
    out["r_i"] = out["r"] - out["i"]
    out["i_z"] = out["i"] - out["z"]

    # Broader colors
    out["u_r"] = out["u"] - out["r"]
    out["g_i"] = out["g"] - out["i"]
    out["r_z"] = out["r"] - out["z"]

    return out

train_fe = add_color_features(train)
test_fe = add_color_features(test)

print("Added features:", COLOR_FEATURES)
display(train_fe[BASE_NUMERIC_COLS + COLOR_FEATURES].head())

Added features: ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z']


,alpha,delta,u,g,r,i,z,redshift,u_g,g_r,r_i,i_z,u_r,g_i,r_z
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,3.576564,1.537632,1.100813,0.636056,5.114196,2.638446,1.736869
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,1.691447,1.499854,0.361141,0.439634,3.191300,1.860995,0.800775
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,-0.043925,-0.092712,0.589211,0.025263,-0.136637,0.496499,0.614474
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,2.254320,2.032982,0.652096,0.450706,4.287302,2.685077,1.102802
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,2.231478,1.237231,0.335002,0.283262,3.468709,1.572233,0.618264


In [5]:
# ============================================================
# 4. Preprocessing
# ============================================================

def prepare_lgb(train_df: pd.DataFrame, test_df: pd.DataFrame):
    X = train_df[FEATURE_COLS].copy()
    X_test = test_df[FEATURE_COLS].copy()

    # LightGBM can consume pandas categorical dtype.
    # Fit category vocabulary on train+test to avoid unseen-category dtype mismatch.
    for col in CATEGORICAL_COLS:
        all_values = pd.concat([X[col], X_test[col]], axis=0).astype("string").fillna("__NA__")
        categories = sorted(all_values.unique().tolist())
        X[col] = pd.Categorical(X[col].astype("string").fillna("__NA__"), categories=categories)
        X_test[col] = pd.Categorical(X_test[col].astype("string").fillna("__NA__"), categories=categories)

    for col in NUMERIC_COLS:
        X[col] = X[col].astype("float32")
        X_test[col] = X_test[col].astype("float32")

    return X, X_test

X, X_test = prepare_lgb(train_fe, test_fe)

le = LabelEncoder()
y = le.fit_transform(train_fe[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

print("class_names:", class_names)
print("n_classes:", n_classes)
print("FEATURE_COLS:", FEATURE_COLS)
print("X dtypes:")
display(X.dtypes.to_frame("dtype"))

print("\\nEncoded target distribution:")
display(pd.Series(y).value_counts().sort_index().rename(index={i: c for i, c in enumerate(class_names)}).to_frame("count"))

class_names: ['GALAXY', 'QSO', 'STAR']
n_classes: 3
FEATURE_COLS: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z', 'spectral_type', 'galaxy_population']
X dtypes:


,dtype
alpha,float32
delta,float32
u,float32
g,float32
r,float32
i,float32
z,float32
redshift,float32
u_g,float32
g_r,float32


\nEncoded target distribution:


,count
GALAXY,377480
QSO,117143
STAR,82724


In [6]:
# ============================================================
# 5. Model params
# ============================================================

# Keep same params as 002 LightGBM strict raw baseline.
# This isolates the effect of minimal color-index FE.
LGB_PARAMS = dict(
    objective="multiclass",
    num_class=n_classes,
    boosting_type="gbdt",
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    force_col_wise=True,
)

FIT_CALLBACKS = [
    lgb.early_stopping(stopping_rounds=150, verbose=False),
    lgb.log_evaluation(period=200),
]

print(json.dumps(LGB_PARAMS, indent=2))

{
  "objective": "multiclass",
  "num_class": 3,
  "boosting_type": "gbdt",
  "n_estimators": 5000,
  "learning_rate": 0.03,
  "num_leaves": 127,
  "max_depth": -1,
  "min_child_samples": 40,
  "subsample": 0.85,
  "subsample_freq": 1,
  "colsample_bytree": 0.85,
  "reg_alpha": 0.0,
  "reg_lambda": 1.0,
  "random_state": 42,
  "n_jobs": -1,
  "verbosity": -1,
  "force_col_wise": true
}


In [7]:
# ============================================================
# 6. CV training
# ============================================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_proba = np.zeros((len(train_fe), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_fe), n_classes), dtype=np.float32)

fold_rows = []
feature_importance_rows = []
models = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = LGBMClassifier(**LGB_PARAMS)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="multi_logloss",
        categorical_feature=CATEGORICAL_COLS,
        callbacks=FIT_CALLBACKS,
    )

    best_iter = model.best_iteration_ or LGB_PARAMS["n_estimators"]

    va_proba = model.predict_proba(X_va, num_iteration=best_iter)
    te_proba = model.predict_proba(X_test, num_iteration=best_iter)

    oof_proba[va_idx] = va_proba.astype(np.float32)
    test_proba += te_proba.astype(np.float32) / N_SPLITS

    va_pred = va_proba.argmax(axis=1)
    fold_score = balanced_accuracy_score(y_va, va_pred)

    fold_rows.append({
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        "best_iteration": int(best_iter),
        "n_train": int(len(tr_idx)),
        "n_valid": int(len(va_idx)),
    })

    fi = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance_gain": model.booster_.feature_importance(importance_type="gain"),
        "importance_split": model.booster_.feature_importance(importance_type="split"),
        "fold": fold,
    })
    feature_importance_rows.append(fi)

    print("fold balanced_accuracy:", fold_score)
    print("best_iteration:", best_iter)
    print("confusion matrix:")
    print(confusion_matrix(y_va, va_pred))

    models.append(model)

fold_scores = pd.DataFrame(fold_rows)
feature_importance = pd.concat(feature_importance_rows, ignore_index=True)

oof_pred = oof_proba.argmax(axis=1)
cv_score = balanced_accuracy_score(y, oof_pred)

print("\n========== CV result ==========")
display(fold_scores)
print("OOF balanced_accuracy:", cv_score)
print("delta vs 002 LGB strict raw:", cv_score - REF_002_CV)
print("\nClassification report:")
print(classification_report(y, oof_pred, target_names=class_names))


========== Fold 1/5 ==========
[200]	valid_0's multi_logloss: 0.0913838
[400]	valid_0's multi_logloss: 0.088094
[600]	valid_0's multi_logloss: 0.0875917
[800]	valid_0's multi_logloss: 0.0876815
fold balanced_accuracy: 0.9562031819787622
best_iteration: 659
confusion matrix:
[[73927   672   897]
 [  635 22602   192]
 [ 1148    98 15299]]

========== Fold 2/5 ==========
[200]	valid_0's multi_logloss: 0.0920424
[400]	valid_0's multi_logloss: 0.0889559
[600]	valid_0's multi_logloss: 0.0885593
fold balanced_accuracy: 0.9579469637084644
best_iteration: 630
confusion matrix:
[[73846   672   978]
 [  598 22635   196]
 [ 1071    94 15380]]

========== Fold 3/5 ==========
[200]	valid_0's multi_logloss: 0.0932356
[400]	valid_0's multi_logloss: 0.0902157
[600]	valid_0's multi_logloss: 0.0897743
fold balanced_accuracy: 0.9571543542633482
best_iteration: 592
confusion matrix:
[[73771   686  1039]
 [  625 22597   207]
 [ 1068    93 15383]]

========== Fold 4/5 ==========
[200]	valid_0's multi_loglos

,fold,balanced_accuracy,best_iteration,n_train,n_valid
0,1,0.956203,659,461877,115470
1,2,0.957947,630,461877,115470
2,3,0.957154,592,461878,115469
3,4,0.956755,588,461878,115469
4,5,0.957173,717,461878,115469


OOF balanced_accuracy: 0.9570463209115138
delta vs 002 LGB strict raw: 0.00013998081126354034

Classification report:
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98    377480
         QSO       0.97      0.96      0.97    117143
        STAR       0.93      0.93      0.93     82724

    accuracy                           0.97    577347
   macro avg       0.96      0.96      0.96    577347
weighted avg       0.97      0.97      0.97    577347



In [8]:
# ============================================================
# 7. Submission
# ============================================================

test_pred = test_proba.argmax(axis=1)
test_labels = le.inverse_transform(test_pred)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: test_labels,
})

# Validate submission schema
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

print(submission[TARGET_COL].value_counts(normalize=True))
display(submission.head())

submission_path = OUTDIR / "submission_lgb_color_index_min.csv"
submission.to_csv(submission_path, index=False)
print("saved:", submission_path)

class
GALAXY    0.655012
QSO       0.202263
STAR      0.142724
Name: proportion, dtype: float64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


saved: /kaggle/working/exp_20260603_006_lgb_color_index_min/submission_lgb_color_index_min.csv


In [9]:
# ============================================================
# 8. Save artifacts
# ============================================================

np.save(OUTDIR / "oof_lgb_color_index_min_proba.npy", oof_proba)
np.save(OUTDIR / "pred_lgb_color_index_min_proba.npy", test_proba)

oof_df = pd.DataFrame({
    ID_COL: train[ID_COL].values,
    "y_true": train[TARGET_COL].astype(str).values,
    "y_pred": le.inverse_transform(oof_pred),
})
for i, cls in enumerate(class_names):
    oof_df[f"proba_{cls}"] = oof_proba[:, i]
oof_df.to_csv(OUTDIR / "oof_lgb_color_index_min.csv", index=False)

pred_df = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "pred_label": test_labels,
})
for i, cls in enumerate(class_names):
    pred_df[f"proba_{cls}"] = test_proba[:, i]
pred_df.to_csv(OUTDIR / "pred_lgb_color_index_min.csv", index=False)

fold_scores.to_csv(OUTDIR / "fold_scores.csv", index=False)

feature_importance_summary = (
    feature_importance
    .groupby("feature", as_index=False)[["importance_gain", "importance_split"]]
    .mean()
    .sort_values("importance_gain", ascending=False)
)
feature_importance.to_csv(OUTDIR / "feature_importance_by_fold.csv", index=False)
feature_importance_summary.to_csv(OUTDIR / "feature_importance.csv", index=False)

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "model": "LightGBM",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "cv_score": float(cv_score),
    "delta_vs_002_lgb_strict_raw": float(cv_score - REF_002_CV),
    "fold_scores": fold_scores.to_dict(orient="records"),
    "class_names": class_names,
    "label_mapping": {str(cls): int(i) for i, cls in enumerate(class_names)},
    "feature_cols": FEATURE_COLS,
    "base_numeric_cols": BASE_NUMERIC_COLS,
    "color_features": COLOR_FEATURES,
    "categorical_cols": CATEGORICAL_COLS,
    "params": LGB_PARAMS,
    "use_original": False,
    "use_fe": True,
    "fe_family": "color_index_min",
    "use_bias_search": False,
    "submission_path": str(submission_path),
}

with open(OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False, default=str)

print("Artifacts saved to:", OUTDIR)
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Artifacts saved to: /kaggle/working/exp_20260603_006_lgb_color_index_min
 - cv_result.json
 - feature_importance.csv
 - feature_importance_by_fold.csv
 - fold_scores.csv
 - oof_lgb_color_index_min.csv
 - oof_lgb_color_index_min_proba.npy
 - pred_lgb_color_index_min.csv
 - pred_lgb_color_index_min_proba.npy
 - submission_lgb_color_index_min.csv


In [10]:
# ============================================================
# 9. Initialize / update oof_registry.csv
# ============================================================

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "LightGBM",
    "feature_family": "color_index_min",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(cv_score),
    "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": False,
    "oof_path": str(OUTDIR / "oof_lgb_color_index_min_proba.npy"),
    "pred_path": str(OUTDIR / "pred_lgb_color_index_min_proba.npy"),
    "submission_path": str(submission_path),
    "role": "main_axis_fe_candidate",
    "status": "completed",
    "keep_hold_drop": "pending",
    "notes": "LightGBM with minimal color-index FE. No original, no bias search.",
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

display(registry.tail())
print("registry saved:", registry_path)

,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260603_006_lgb_color_index_min,LightGBM,color_index_min,balanced_accuracy_score,0.957046,0.000572,NaN,False,True,False,/kaggle/working/exp_20260603_006_lgb_color_ind...,/kaggle/working/exp_20260603_006_lgb_color_ind...,/kaggle/working/exp_20260603_006_lgb_color_ind...,main_axis_fe_candidate,completed,pending,LightGBM with minimal color-index FE. No origi...


registry saved: /kaggle/working/oof_registry.csv


In [11]:
# ============================================================
# 10. memo.yml
# ============================================================

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "LightGBM color index minimal FE",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "LightGBM",
        "created_at": "2026-06-03",
    },
    "objective": {
        "summary": (
            "Test minimal astronomical color-index features on top of the 002 LightGBM strict raw baseline. "
            "Use only competition train/test. No original dataset, no bias search, and no parameter tuning."
        ),
        "success_criteria": [
            "save oof proba",
            "save test pred proba",
            "save cv_result",
            "save submission",
            "save feature importance",
            "update oof_registry",
            "compare CV against 002 LightGBM strict raw baseline",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "use_original": False,
    },
    "features": {
        "feature_family": "color_index_min",
        "base_numeric_cols": BASE_NUMERIC_COLS,
        "color_features": COLOR_FEATURES,
        "categorical_cols": CATEGORICAL_COLS,
        "feature_cols": FEATURE_COLS,
        "notes": (
            "Adds only color-index differences: u-g, g-r, r-i, i-z, u-r, g-i, r-z. "
            "No redshift bin, no ratio, no original prior, no snap."
        ),
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "score": float(cv_score),
        "delta_vs_002_lgb_strict_raw": float(cv_score - REF_002_CV),
        "reference_002_cv": REF_002_CV,
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    },
    "params": LGB_PARAMS,
    "outputs": {
        "oof_proba": "oof_lgb_color_index_min_proba.npy",
        "pred_proba": "pred_lgb_color_index_min_proba.npy",
        "oof_csv": "oof_lgb_color_index_min.csv",
        "pred_csv": "pred_lgb_color_index_min.csv",
        "submission": "submission_lgb_color_index_min.csv",
        "cv_result": "cv_result.json",
        "feature_importance": "feature_importance.csv",
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb",
        "reason": (
            "This is the first minimal FE experiment after raw baselines. "
            "Adoption depends on CV delta vs 002, Public LB, and later OOF correlation."
        ),
        "next_action": [
            "Submit submission_lgb_color_index_min.csv",
            "Record Public LB",
            "If CV/LB improves, transfer same FE to XGBoost",
            "If it does not improve, do not expand color FE blindly",
            "Do not start Optuna yet",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("memo saved:", memo_path)

memo saved: /kaggle/working/exp_20260603_006_lgb_color_index_min/memo.yml
